In [5]:
import os, requests, time

WEBHOOK_URL = os.getenv('DISCORD_WEBHOOK_URL_TEST')

def notify_discord(message, job_finished=True):
    try:

        timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
        payload = {
            "content": f"{'*Job finished * \n📅'*job_finished} {timestamp}\n{message}"
        }
        requests.post(WEBHOOK_URL, json=payload, timeout=10)
    except Exception as e:
        print("Could not send Discord notification:", e)

In [6]:
notify_discord("test")

Could not send Discord notification: Invalid URL 'None': No scheme supplied. Perhaps you meant https://None?


In [1]:
import torch
torch.cuda.is_available()

True

In [43]:
import torch
import pickle
import tensorflow as tf
from utils import DATA_DIR

import tensorly as tl
from tensorly.decomposition import tucker, non_negative_tucker, non_negative_tucker_hals
import time
from typing import List, Tuple


# we load the tensor and vocabularies
# we save the respective vocabularies as well
# vocab = {
#     "vocab_v": vocab_v,
#     "vocab_s": vocab_s,
#     "vocab_o": vocab_o,
#     "v2i": v2i,
#     "s2i": s2i,
#     "o2i": o2i
# }
with open(DATA_DIR/"tensors/fineweb_dutch_vectors_ids/vocabularies_1000.pkl", "rb") as f:
    vocab = torch.load(f)
vocab_v = vocab["vocab_v"]
vocab_s = vocab["vocab_s"]
vocab_o = vocab["vocab_o"]
v2i = vocab["v2i"]
s2i = vocab["s2i"]
o2i = vocab["o2i"]

def _torch_or_tucker_load(path, map_location="cpu"):
    """Tries to load a torch-saved file, if fails, tries pickle."""
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except RuntimeError:
        with open(path, "rb") as f:
            return pickle.load(f)

class TupleTensor:

    def __init__(self, path: str):
        self.tensor = _torch_or_tucker_load(path)


    def sparse_representation(self):
        # we return the sparse representation of the tensor
        # we check if our tensor is a tensorflow tensor or make it one
        if not isinstance(self.tensor, tf.Tensor):
            self.tensor = tf.convert_to_tensor(self.tensor)
        return tf.sparse.from_dense(self.tensor)

    def tensor_to_sparse(self):
        self.tensor = self.sparse_representation()

    def decompose(self, non_negative=True, hals=True, rank=None,
                  n_iter_max=100, init="random", tol=1e-12, verbose=False):
        if rank is None:
            rank = [100, 100, 100]
        tic = time.time()

        if not non_negative:
            decomp = tucker
        elif hals:
            decomp = non_negative_tucker_hals
        else:
            decomp = non_negative_tucker

        core, factors = decomp(self.tensor, rank=rank, n_iter_max=n_iter_max,
                               init=init, tol=tol, verbose=verbose)
        toc = time.time()
        print("Time taken for decomposition", toc-tic)
        return core, factors

tensor = TupleTensor(DATA_DIR/"tensors/fineweb_dutch_vectors_ids/tensor_sii_1000.pkl")


In [44]:
nnt_sparse(sparse_tensor_tensorly, rank=[100,100,100], n_iter_max=100, verbose=True)

KeyboardInterrupt: 

In [21]:
# we save the sparse tensor to disk to check its size
with open(DATA_DIR/"tensors/fineweb_dutch_vectors_ids/tensor_sii_1000_sparse.pkl", "wb") as f:
    pickle.dump(sparse_tensor, f)

# we check if we can go back to a dense tensor
dense_tensor = tf.sparse.to_dense(sparse_tensor)
# we check if they are the same

Tensors are equal: True


In [3]:
£import os
import tensorly as tl

os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # 0 is being used at time of writing
tl.set_backend('pytorch')
torch.cuda.set_device(0)                       # now device 0 maps to physical GPU 1

In [ ]:
# we perform Tucker Decomposition using tensorly
from tensorly.decomposition import tucker, non_negative_tucker, non_negative_tucker_hals
import time

# ensure float32 and move to GPU (if they are numpy arrays)
tensor_counting = torch.as_tensor(tensor_counting, dtype=torch.float32, device='cuda')
tensor_sc  = torch.as_tensor(tensor_sc,  dtype=torch.float32, device='cuda')
tensor_sii = torch.as_tensor(tensor_sii, dtype=torch.float32, device='cuda')

# (optional) faster matmul kernels on PyTorch 2.x
try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

def _to_np(x):
    # Accept NumPy arrays or torch tensors; return NumPy view/copy
    if hasattr(x, "detach"):  # torch.Tensor
        return x.detach().cpu().numpy()
    return x

def _torch_or_tucker_load(path, map_location="cpu"):
    """Tries to load a torch-saved file, if fails, tries pickle."""
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except RuntimeError:
        with open(path, "rb") as f:
            return pickle.load(f)



In [16]:


def decompose(tensor, non_negative=True, hals=True, rank=None,
              n_iter_max=100, init="random", tol=1e-12, verbose=False):
    if rank is None:
        rank = [100, 100, 100]
    tic = time.time()

    if not non_negative:
        decomp = tucker
    elif hals:
        decomp = non_negative_tucker_hals
    else:
        decomp = non_negative_tucker

    core, factors = decomp(tensor, rank=rank, n_iter_max=n_iter_max,
                           init=init, tol=tol, verbose=verbose)
    toc = time.time()
    print("Time taken for decomposition", toc-tic)
    return core, factors

# some shenanigans to show iteration numbers live
import sys, io, re
from contextlib import contextmanager

_NUM = r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?"

class IterPrefixer(io.TextIOBase):
    def __init__(self, orig, on_iter=None, autoflush=True):
        self.orig = orig
        self.buf = ""
        self.iter = 0
        self.on_iter = on_iter
        self.autoflush = autoflush
        # NOTE: no lookahead at the end; we'll strip punctuation when parsing
        self._regex = re.compile(
            rf"reconstruction\s*error\s*=\s*({_NUM})\s*,\s*variation\s*=\s*({_NUM})",
            re.IGNORECASE,
        )

    def write(self, s):
        self.buf += s
        # robust line handling (supports mixed \n / \r\n / partial lines)
        while True:
            nl = self.buf.find("\n")
            if nl == -1:
                break
            line = self.buf[:nl]
            self.buf = self.buf[nl+1:]

            m = self._regex.search(line)
            if m:
                self.iter += 1
                err_str = m.group(1).rstrip(".,;: ")
                var_str = m.group(2).rstrip(".,;: ")
                try:
                    err = float(err_str)
                    var = float(var_str)
                except ValueError:
                    err = var = None  # fail-safe
                if self.on_iter and (err is not None) and (var is not None):
                    try:
                        self.on_iter(self.iter, err, var)
                    except Exception:
                        pass
                line = f"[iter {self.iter:03d}] " + line

            self.orig.write(line + "\r")
            if self.autoflush:
                try: self.orig.flush()
                except Exception: pass
        return len(s)

    def flush(self):
        if self.buf:
            # write any trailing partial line as-is
            self.orig.write(self.buf)
            self.buf = "\r"
        try: self.orig.flush()
        except Exception: pass

@contextmanager
def live_iteration_numbers(on_iter=None):
    old_stdout, old_stderr = sys.stdout, sys.stderr
    pref_out = IterPrefixer(old_stdout, on_iter=on_iter, autoflush=True)
    pref_err = IterPrefixer(old_stderr, on_iter=on_iter, autoflush=True)
    sys.stdout, sys.stderr = pref_out, pref_err
    try:
        yield
    finally:
        sys.stdout, sys.stderr = old_stdout, old_stderr

In [8]:
with live_iteration_numbers():
    core_sc, factors_sc = decompose(tensor_sc, hals=False, init="random", verbose=True, n_iter_max=1000)
    core_sc_svd, factors_sc_svd = decompose(tensor_sc, hals=False, init="svd", verbose=True, n_iter_max=1000)
tucker_reconstruction_mu = tl.tucker_to_tensor((core_sc, factors_sc))
tucker_reconstruction_svd = tl.tucker_to_tensor((core_sc_svd, factors_sc_svd))
error_mu = tl.norm(tensor_sc - tucker_reconstruction_mu) / tl.norm(tensor_sc)
error_svd = tl.norm(tensor_sc - tucker_reconstruction_svd) / tl.norm(tensor_sc)
print("Relative error (random init):", error_mu.item())
print("Relative error (svd init):", error_svd.item())

[iter 001] reconstruction error=0.9091246724128723, variation=0.003446042537689209.
[iter 002] reconstruction error=0.908414363861084, variation=0.0007103085517883301.
[iter 003] reconstruction error=0.908227264881134, variation=0.00018709897994995117.
[iter 004] reconstruction error=0.9081282615661621, variation=9.900331497192383e-05.
[iter 005] reconstruction error=0.9080296158790588, variation=9.864568710327148e-05.
[iter 006] reconstruction error=0.907907247543335, variation=0.00012236833572387695.
[iter 007] reconstruction error=0.9077455401420593, variation=0.00016170740127563477.
[iter 008] reconstruction error=0.9075254201889038, variation=0.00022011995315551758.
[iter 009] reconstruction error=0.9072167873382568, variation=0.00030863285064697266.
[iter 010] reconstruction error=0.9067718386650085, variation=0.000444948673248291.
[iter 011] reconstruction error=0.9061117172241211, variation=0.0006601214408874512.
[iter 012] reconstruction error=0.9051052927970886, variation=0.0

In [10]:
core_sc_hals, factors_sc_hals = decompose(tensor_sc)
tucker_reconstruction_sc_hals = tl.tucker_to_tensor((core_sc_hals, factors_sc_hals))

Time taken for Non-negative Tucker decomposition: 469.73478412628174


In [19]:
with live_iteration_numbers():
    core_sii, factors_sii = decompose(tensor_sii, hals=False, verbose=True, init="svd")
tucker_reconstruction_sii = tl.tucker_to_tensor((core_sii, factors_sii))

OutOfMemoryError: CUDA out of memory. Tried to allocate 3.68 GiB. GPU 0 has a total capacity of 23.56 GiB of which 2.22 GiB is free. Including non-PyTorch memory, this process has 21.33 GiB memory in use. Of the allocated memory 16.61 GiB is allocated by PyTorch, and 4.38 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [22]:
core_sii_hals, factors_sii_hals = decompose(tensor_sii)
tucker_reconstruction_sii_hals = tl.tucker_to_tensor((core_sii_hals, factors_sii_hals))

Time taken for decomposition 462.8394057750702


In [24]:
# we save to a pickle
with open(DATA_DIR/"tensors/tucker_decomposition_sc_100d_100i.pkl", "wb") as f:
    pickle.dump((core_sc, factors_sc), f)
with open(DATA_DIR/"tensors/tucker_decomposition_sii_100d_1000i.pkl", "wb") as f:
    pickle.dump((core_sii, factors_sii), f)
with open(DATA_DIR/"tensors/tucker_decomposition_hals_sc_100d_100i.pkl", "wb") as f:
    pickle.dump((core_sc_hals, factors_sc_hals), f)
with open(DATA_DIR/"tensors/tucker_decomposition_hals-sii_100d_100i.pkl", "wb") as f:
    pickle.dump((core_sii_hals, factors_sii_hals), f)

In [49]:
core_count_hals, factors_count_hals = decompose(tensor_counting, verbose=True)
with open(DATA_DIR/"tensors/tucker_decomposition_hals_count_100d_100i.pkl", "wb") as f:
    pickle.dump((core_count_hals, factors_count_hals), f)

reconstruction error=0.18893150985240936, variation=0.048352986574172974.
reconstruction error=0.15962429344654083, variation=0.02930721640586853.
reconstruction error=0.14270161092281342, variation=0.016922682523727417.
reconstruction error=0.13334347307682037, variation=0.009358137845993042.
reconstruction error=0.12717971205711365, variation=0.006163761019706726.
reconstruction error=0.12185388803482056, variation=0.005325824022293091.
reconstruction error=0.11633662134408951, variation=0.005517266690731049.
reconstruction error=0.11067275702953339, variation=0.005663864314556122.
reconstruction error=0.10600420832633972, variation=0.0046685487031936646.
reconstruction error=0.1025015264749527, variation=0.003502681851387024.
reconstruction error=0.10016163438558578, variation=0.002339892089366913.
reconstruction error=0.0985536202788353, variation=0.0016080141067504883.
reconstruction error=0.09743569046258926, variation=0.0011179298162460327.
reconstruction error=0.096556350588798

In [61]:
with live_iteration_numbers():
    core_sc, factors_sc = decompose(
        tensor_sc,
        non_negative=False,
        hals=False,
        rank=[100,100,100],
        n_iter_max=100,
        verbose=True
    )


[iter 001] reconstruction error=0.6090680956840515, variation=0.001820385456085205.
[iter 002] reconstruction error=0.6086451411247253, variation=0.0004229545593261719.
[iter 003] reconstruction error=0.6087465286254883, variation=-0.00010138750076293945.
[iter 004] reconstruction error=0.6086974740028381, variation=4.9054622650146484e-05.
[iter 005] reconstruction error=0.6085110902786255, variation=0.00018638372421264648.
[iter 006] reconstruction error=0.6085413694381714, variation=-3.0279159545898438e-05.
[iter 007] reconstruction error=0.6085367798805237, variation=4.589557647705078e-06.
[iter 008] reconstruction error=0.6085678339004517, variation=-3.1054019927978516e-05.
[iter 009] reconstruction error=0.6086316108703613, variation=-6.377696990966797e-05.
[iter 010] reconstruction error=0.608640193939209, variation=-8.58306884765625e-06.
[iter 011] reconstruction error=0.6085934638977051, variation=4.673004150390625e-05.
[iter 012] reconstruction error=0.6085808277130127, variat

In [25]:
# we check average error:
error = tl.norm(tensor_sc - tucker_reconstruction) / tl.norm(tensor_sc)
error_non_negative_sc = tl.norm(tensor_sc - tucker_reconstruction_mu) / tl.norm(tensor_sc)
error_non_negative_sii = tl.norm(tensor_sii - tucker_reconstruction_sii) / tl.norm(tensor_sii)
error_non_negative_sc_hals = tl.norm(tensor_sc - tucker_reconstruction_sc_hals) / tl.norm(tensor_sc)
error_non_negative_sii_hals = tl.norm(tensor_sii - tucker_reconstruction_sii_hals) / tl.norm(tensor_sii)
print("Relative error (Tucker SC):", error.item())
print("Relative error (Non-negative Tucker SC):", error_non_negative_sc.item())
print("Relative error (Non-negative Tucker with HALS instead of HOOI, SC", error_non_negative_sc_hals.item())
print("Relative error (Non-negative Tucker SII):", error_non_negative_sii.item())
print("Relative error (Non-negative Tucker with HALS instead of HOOI, SII", error_non_negative_sii_hals.item())


Relative error (Tucker): 0.609062910079956
Relative error (Non-negative Tucker SC): 0.7910473346710205
Relative error (Non-negative Tucker with HALS instead of HOOI, SC 0.796667754650116
Relative error (Non-negative Tucker SII): 0.7481531500816345
Relative error (Non-negative Tucker with HALS instead of HOOI, SII 0.8195702433586121


In [38]:
# we try to interpret the factors ~ latent dimensions
# we normalize the factors for easier interpretation

# we first move the tensor back to cpu
factor_verb = factors_sc_hals[0].cpu()
factor_subj = factors_sc_hals[1].cpu()
factor_obj = factors_sc_hals[2].cpu()
core = core_sc_hals.cpu()

In [31]:
for i in range(factor_verb.shape[1]):
    print(f"Dimension {i}:")
    # verbs
    top_values, top_indices = torch.topk(factor_verb[:, i], 10)
    top_verbs = [f"{vocab_v[idx]} ({top_values[j].item():.3f})" for j, idx in enumerate(top_indices)]
    # subjects
    top_values, top_indices = torch.topk(factor_subj[:, i], 10)
    top_subj = [f"{vocab_s[idx]} ({top_values[j].item():.3f})" for j, idx in enumerate(top_indices)]
    # objects
    top_values, top_indices = torch.topk(factor_obj[:, i], 10)
    top_obj = [f"{vocab_o[idx]} ({top_values[j].item():.3f})" for j, idx in enumerate(top_indices)]
    
    print("verbs:", top_verbs)
    print("subjects:", top_subj)
    print("objects:", top_obj)
    print("\n")

Dimension 0:
verbs: ['denken (0.000)', 'zijn (0.000)', 'doen (0.000)', 'laten (0.000)', 'hebben (0.000)', 'weten (0.000)', 'willen (0.000)', 'gaan (0.000)', 'komen (0.000)', 'zeggen (0.000)']
subjects: ['ik (4410.667)', '~ (2668.086)', 'je (2136.206)', 'we (896.877)', 'u (277.364)', 'hij (61.024)', 'jij (6.145)', 'jezelf (0.735)', 'spoorweg (0.414)', 'uitweg (0.238)']
objects: ['me (758.133)', 'ons (405.009)', 'mij (306.431)', 'je (93.880)', 'man (70.075)', 'mens (52.546)', 'elkaar (47.530)', 'me. (46.948)', 'iemand (41.192)', 'jezelf (31.783)']


Dimension 1:
verbs: ['wegsturen (0.000)', 'dagen (0.000)', 'help (0.000)', 'lokken (0.000)', 'jaag (0.000)', 'waarschuw (0.000)', 'vang (0.000)', 'schenk (0.000)', 'bestuderen (0.000)', 'schudden (0.000)']
subjects: ['hij (1825.240)', 'ze (1739.903)', 'u (1646.644)', 'we (1171.871)', 'je (1104.528)', 'jij (827.656)', 'jullie (494.370)', '~ (310.157)', 'wij (287.613)', 'wie (238.711)']
objects: ['geld (154.353)', 'veel (132.759)', 'alles (121.

In [32]:
# based on this idea, we can build up a cosine similarity function
from sklearn.metrics.pairwise import cosine_similarity

def verb_similarity(verb1, verb2):
    if verb1 not in v2i or verb2 not in v2i:
        print("One or both verbs not in vocabulary.")
        return None
    vec1 = factor_verb[v2i[verb1], :].unsqueeze(0).numpy()
    vec2 = factor_verb[v2i[verb2], :].unsqueeze(0).numpy()
    similarity = cosine_similarity(vec1, vec2)[0][0]
    return similarity

# we test it
print("Similarity between 'spelen' and 'winnen':", verb_similarity("spelen", "winnen"))
print("Similarity between 'spelen' and 'oproepen':", verb_similarity("spelen", "oproepen"))
print("Similarity between 'spelen' and 'verliezen':", verb_similarity("spelen", "verliezen"))
print("Similarity between 'winnen' and 'verliezen':", verb_similarity("winnen", "verliezen"))

Similarity between 'spelen' and 'winnen': 0.6092653
Similarity between 'spelen' and 'oproepen': 0.41020817
Similarity between 'spelen' and 'verliezen': 0.18270153
Similarity between 'winnen' and 'verliezen': 0.5392559


In [33]:
# we create a "top similar verbs" function
def top_similar_verbs(target_verb, top_n=5):
    if target_verb not in v2i:
        print(f"Verb '{target_verb}' not in vocabulary.")
        return []
    target_vector = factor_verb[v2i[target_verb], :].unsqueeze(0).numpy()
    similarities = []
    for verb in vocab_v:
        if verb == target_verb:
            continue
        verb_vector = factor_verb[v2i[verb], :].unsqueeze(0).numpy()
        sim = cosine_similarity(target_vector, verb_vector)[0][0]
        similarities.append((verb, sim))
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_n]


In [62]:
# we test it
test = ["spelen", "winnen", "verliezen", "geven", "nemen", "steken", "eisen", "leven"]
for verb in test:
    print(f"Top similar verbs to '{verb}':", top_similar_verbs(verb, 5))

Top similar verbs to 'spelen': [('hoeven', np.float32(0.82185066)), ('kunnen', np.float32(0.7908659)), ('zorgen', np.float32(0.78526646)), ('sterven', np.float32(0.7841965)), ('vertrekken', np.float32(0.7758652))]
Top similar verbs to 'winnen': [('stoppen', np.float32(0.7649036)), ('kosten', np.float32(0.7527344)), ('vergeetlen', np.float32(0.7425939)), ('raken', np.float32(0.73326993)), ('steken', np.float32(0.73097074))]
Top similar verbs to 'verliezen': [('verspillen', np.float32(0.9205954)), ('gebruiken', np.float32(0.8643547)), ('sparen', np.float32(0.8457716)), ('uitgeven', np.float32(0.81289256)), ('bieden', np.float32(0.7846847))]
Top similar verbs to 'geven': [('krijgen', np.float32(0.99824125)), ('ontvangen', np.float32(0.6480978)), ('bezorgen', np.float32(0.5553591)), ('sturen', np.float32(0.45381922)), ('nemen', np.float32(0.36890435))]
Top similar verbs to 'nemen': [('brengen', np.float32(0.5848553)), ('pak', np.float32(0.5492506)), ('dragen', np.float32(0.53501177)), ('ko

# Core tensor shift modelling
By obtaining the dimension (k) values of each mode, we can retrace the appropriate point in the core tensor
Then, we can calculate the "distance" between two sentences in this core tensor as a "shift" in meaning

## Effect of normalizing the data
There is an effect: In the weighted version, some quantile information gets changed (we already normalize in the function)

In [40]:
import itertools
import numpy as np
import plotly.graph_objects as go

# we move to cpu

core = core.numpy()

def create_core_projection(test_tuple, weighted=False):
    verb_idx = v2i[test_tuple[0]]
    subj_idx = s2i[test_tuple[1]]
    obj_idx = o2i[test_tuple[2]]
    # we get their corresponding values in each dimension
    verb_dims = factor_verb[verb_idx, :].numpy()
    subj_dims = factor_subj[subj_idx, :].numpy()
    obj_dims = factor_obj[obj_idx, :].numpy()
    
    # we project this to a distribution in a core tensor like space
    core_projection = np.zeros(core.shape)
    for i, j, k in itertools.product(range(core.shape[0]), range(core.shape[1]), range(core.shape[2])):
        core_projection[i, j, k] = verb_dims[i] * subj_dims[j] * obj_dims[k]
    
    if weighted:
        core_projection = core_projection * core
    return core_projection
# Using some algebraic ideas, we can make the same function a lot faster
# --> Not actually populating the core tensor element by element, but using Einstein summation (implemented via np)
def core_projection_fast(test_tuple, weighted=True):
    verb_idx = v2i[test_tuple[0]]
    subj_idx = s2i[test_tuple[1]]
    obj_idx = o2i[test_tuple[2]]
    v = factor_verb[verb_idx, :].cpu().numpy()
    s = factor_subj[subj_idx, :].cpu().numpy()
    o = factor_obj[obj_idx, :].cpu().numpy()
    if weighted:
        # elementwise multiply the core by the outer-product weights
        # shape (R1,R2,R3): v[i]*s[j]*o[k] * core[i,j,k]
        proj = np.einsum('i,j,k,ijk->ijk', v, s, o, core)
    else:
        proj = np.einsum('i,j,k->ijk', v, s, o)
    return proj

def plot_tensor_plotly(tensor: np.ndarray,
                       title=None,
                       top_quantile=0.95,
                       n_isosurfaces=10,
                       show_cube=True,
                       show_grid=True,
                       cube_opacity=0.1,
                       show=True,
                       save_dir=None,
                       return_fig=False):
    """
    Render a 3D tensor as a transparent cube with only the highest values visible.

    - tensor: 3D numpy array (Z, Y, X) or (X, Y, Z) — any order is fine for scalar values
    - top_quantile: only values >= this quantile get an isosurface
    - n_isosurfaces: number of nested high-value surfaces to draw
    """
    assert tensor.ndim == 3, "tensor must be 3D"

    # Normalize to [0, 1] (keeps relative structure)
    vol = tensor.astype(np.float32)
    # vmin, vmax = float(np.nanmin(vol)), float(np.nanmax(vol))
    # if vmax == vmin:
    #     raise ValueError("Tensor has constant value; nothing to visualize.")
    # vol = (vol - vmin) / (vmax - vmin)

    # Choose isovals in the top tail
    q0 = np.quantile(vol, top_quantile)
    # Distribute a few levels between q0 and 1.0
    isovals = np.linspace(q0, 1.0, n_isosurfaces)

    Z, Y, X = vol.shape  # plotly expects x,y,z arrays but can index by grid
    fig = go.Figure()

    # Optional: translucent wireframe cube for context
    if show_cube:
        # Cube edges as line segments
        corners = np.array([
            [0, 0, 0], [X-1, 0, 0], [X-1, Y-1, 0], [0, Y-1, 0],
            [0, 0, Z-1], [X-1, 0, Z-1], [X-1, Y-1, Z-1], [0, Y-1, Z-1]
        ])
        edges = [
            (0,1),(1,2),(2,3),(3,0),
            (4,5),(5,6),(6,7),(7,4),
            (0,4),(1,5),(2,6),(3,7)
        ]
        xs, ys, zs = [], [], []
        for i,j in edges:
            xs += [corners[i,0], corners[j,0], None]
            ys += [corners[i,1], corners[j,1], None]
            zs += [corners[i,2], corners[j,2], None]
        fig.add_trace(go.Scatter3d(
            x=xs, y=ys, z=zs,
            mode="lines",
            line=dict(width=2),
            opacity=cube_opacity,
            hoverinfo="skip",
            showlegend=False
        ))

    # High-value isosurfaces
    # Plotly’s Isosurface can take the volume and draw surfaces at given isovalues.
    fig.add_trace(go.Isosurface(
        value=vol.flatten(order="C"),
        x=np.repeat(np.arange(X), Y*Z),
        y=np.tile(np.repeat(np.arange(Y), Z), X),
        z=np.tile(np.arange(Z), X*Y),
        isomin=isovals[0],
        isomax=isovals[-1],
        surface_count=n_isosurfaces,
        # Keep low values invisible; only top tail draws surfaces
        caps=dict(x_show=False, y_show=False, z_show=False),
        opacity=0.3,
        showscale=True
    ))

    fig.update_layout(
        scene=dict(
            xaxis=dict(visible=show_grid),
            yaxis=dict(visible=show_grid),
            zaxis=dict(visible=show_grid),
            aspectmode="data"
        ),
        margin=dict(l=0, r=0, t=0, b=0),   
    )
    if title:
        fig.update_layout(title=str(title))
        # we center it
        fig.update_layout(title_y=0.95)
    if show:
        fig.show()
    if save_dir:
        fig.write_html(save_dir)
    if return_fig:
        return fig
    
# we check the equivalence of both core projection functions
test_tuple = ("spelen", "kind", "bal")
core_proj_slow = create_core_projection(test_tuple, weighted=True)
core_proj_fast = core_projection_fast(test_tuple, weighted=True)
# we use np is all close
print("Core projections equal:", np.allclose(core_proj_slow, core_proj_fast))

Core projections equal: True


In [42]:
variant_sentences = [("spelen", "man", "rol"), 
                     ("spelen", "man", "spel"), 
                     ("spelen", "kind", "rol"), 
                     ("spelen", "kind", "spel")]

for i, variant in enumerate(variant_sentences):
    core_proj = core_projection_fast(variant)
    core_proj_weighted = core_projection_fast(variant, weighted=True)
    # plot_tensor_plotly(core_proj, title=variant, top_quantile=0.5, n_isosurfaces=100, show_cube=True, show=False, cube_opacity=0.3, save_dir=f"core_projection_{i}.html")
    plot_tensor_plotly(core_proj_weighted, title=variant, top_quantile=0.2, n_isosurfaces=1000, show_cube=True, show=False, cube_opacity=0.3, save_dir=fDATA_DIR/"vis/core_projection_{i}_weighted_all.html")

In [64]:
core_0 = core_projection_fast(variant_sentences[0], weighted=True)
core_1 = core_projection_fast(variant_sentences[1], weighted=True)
# we calculate the difference, but only over the top quantile
core_diff_0_1 = core_0 - core_1
plot_tensor_plotly(core_diff_0_1, title=f"Difference between {variant_sentences[0]} and {variant_sentences[1]}",
                   top_quantile=0.5, n_isosurfaces=100, show_cube=True, show=False, cube_opacity=0.3, save_dir=DATA_DIR/"vis/core_difference_0_1.html")
core_diff_1_0 = core_1 - core_0
plot_tensor_plotly(core_diff_1_0, title=f"Difference between {variant_sentences[1]} and {variant_sentences[0]}",
                   top_quantile=0.5, n_isosurfaces=100, show_cube=True, show=False, cube_opacity=0.3, save_dir=DATA_DIR/"vis/core_difference_1_0.html")

In [65]:
core_2 = core_projection_fast(variant_sentences[2], weighted=True)
core_3 = core_projection_fast(variant_sentences[3], weighted=True)
# we calculate the difference, but only over the top quantile
core_diff_2_3 = core_2 - core_3
plot_tensor_plotly(core_diff_2_3, title=f"Difference between {variant_sentences[2]} and {variant_sentences[3]}",
                   top_quantile=0.5, n_isosurfaces=100, show_cube=True, show=False, cube_opacity=0.3, save_dir=DATA_DIR/"vis/core_difference_2_3.html")
core_diff_3_2 = core_3 - core_2
plot_tensor_plotly(core_diff_3_2, title=f"Difference between {variant_sentences[3]} and {variant_sentences[2]}",
                   top_quantile=0.5, n_isosurfaces=100, show_cube=True, show=False, cube_opacity=0.3, save_dir=DATA_DIR/"vis/core_difference_3_2.html")

In [66]:
from numpy.linalg import norm
cores = [core_0, core_1, core_2, core_3]
for i in range(len(cores)):
    for j in range(i + 1, len(cores)):
        core0 = cores[i]
        core1 = cores[j]
        
        print(f"Comparing core {i} and core {j}:")
        print(f"Sentences: {variant_sentences[i]} vs {variant_sentences[j]}")
        # Flatten tensors
        v0 = core0.flatten()
        v1 = core1.flatten()
        
        # Basic measures
        euclidean = norm(v0 - v1)
        cosine = np.dot(v0, v1) / (norm(v0) * norm(v1))
        manhattan = np.sum(np.abs(v0 - v1))
        relative_change = np.mean(np.abs((v0 - v1) / (v1 + 1e-9)))  # gpt created extension
        
        print("Euclidean:", euclidean)
        print("Cosine similarity:", cosine)
        print("Manhattan:", manhattan)
        print("Relative change:", relative_change)
        # the relative difference is not symmetric, we do it again for the other way around
        relative_change_rev = np.mean(np.abs((v1 - v0) / (v0 + 1e-9)))  # gpt created extension
        print("Reverse relative change:", relative_change_rev)
        print()


Comparing core 0 and core 1:
Sentences: ('spelen', 'man', 'rol') vs ('spelen', 'man', 'spel')
Euclidean: 0.019509569
Cosine similarity: 0.6920675
Manhattan: 0.07908578
Relative change: 21.08276
Reverse relative change: 9.24316

Comparing core 0 and core 2:
Sentences: ('spelen', 'man', 'rol') vs ('spelen', 'kind', 'rol')
Euclidean: 0.015210104
Cosine similarity: 0.10556245
Manhattan: 0.05679988
Relative change: 44.589977
Reverse relative change: 6.868327

Comparing core 0 and core 3:
Sentences: ('spelen', 'man', 'rol') vs ('spelen', 'kind', 'spel')
Euclidean: 0.015258055
Cosine similarity: 0.11580273
Manhattan: 0.05984398
Relative change: 45.626087
Reverse relative change: 7.940122

Comparing core 1 and core 2:
Sentences: ('spelen', 'man', 'spel') vs ('spelen', 'kind', 'rol')
Euclidean: 0.026480485
Cosine similarity: 0.1792819
Manhattan: 0.08446326
Relative change: 61.66291
Reverse relative change: 8.100049

Comparing core 1 and core 3:
Sentences: ('spelen', 'man', 'spel') vs ('spelen',

## Core projection difference
For a target tuple, calculate which "contextualized" verb would be most similar
--> We calculate all core projections for all verbs with the same subject and object, and see which is most similar to the target core projection

In [68]:
from tqdm import tqdm
def contextually_similar_verb(target_tuple, distance="cosine"):
    target_core = core_projection_fast(target_tuple, weighted=True)
    subj = target_tuple[1]
    obj = target_tuple[2]
    
    similarities = []
    for verb in tqdm(vocab_v):
        if verb == target_tuple[0]:
            continue
        test_tuple = (verb, subj, obj)
        test_core = core_projection_fast(test_tuple, weighted=True)
        
        if distance == "cosine":
            # Flatten tensors
            v0 = target_core.flatten()
            v1 = test_core.flatten()
            
            # Cosine similarity
            cosine = np.dot(v0, v1)
            similarities.append((verb, cosine))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities

def contextually_similar_subj(target_tuple, distance="cosine"):
    target_core = core_projection_fast(target_tuple, weighted=True)
    verb = target_tuple[0]
    obj = target_tuple[2]
    
    similarities = []
    for subj in tqdm(vocab_s):
        if subj == target_tuple[1]:
            continue
        test_tuple = (verb, subj, obj)
        test_core = core_projection_fast(test_tuple, weighted=True)
        
        if distance == "cosine":
            # Flatten tensors
            v0 = target_core.flatten()
            v1 = test_core.flatten()
            
            # Cosine similarity
            cosine = np.dot(v0, v1) # / (norm(v0) * norm(v1))
            similarities.append((subj, cosine))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities

def contextually_similar_obj(target_tuple, distance="cosine"):
    target_core = core_projection_fast(target_tuple, weighted=True)
    verb = target_tuple[0]
    subj = target_tuple[1]
    
    similarities = []
    for obj in tqdm(vocab_o):
        if obj == target_tuple[2]:
            continue
        test_tuple = (verb, subj, obj)
        test_core = core_projection_fast(test_tuple, weighted=True)
        
        if distance == "cosine":
            # Flatten tensors
            v0 = target_core.flatten()
            v1 = test_core.flatten()
            
            # Cosine similarity
            cosine = np.dot(v0, v1) # / (norm(v0) * norm(v1))
            similarities.append((obj, cosine))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities

# we test with a variant sentence
contextualised_distance = contextually_similar_verb(("spelen", "kind", "spel"))
verb_distance = top_similar_verbs("spelen", top_n=10)
print("Contextually similar verbs (core projection distance):", contextualised_distance[:10])
print("General similar verbs (factor cosine similarity):", verb_distance)

100%|██████████| 750/750 [00:02<00:00, 266.74it/s]


Contextually similar verbs (core projection distance): [('doen', np.float32(7.9739024e-05)), ('horen', np.float32(2.4958266e-05)), ('maken', np.float32(2.2161812e-05)), ('kennen', np.float32(1.8148743e-05)), ('lezen', np.float32(1.7770373e-05)), ('weten', np.float32(1.769957e-05)), ('houden', np.float32(1.3228166e-05)), ('vertellen', np.float32(1.3091114e-05)), ('schrijven', np.float32(1.2918729e-05)), ('vinden', np.float32(1.2694761e-05))]
General similar verbs (factor cosine similarity): [('hoeven', np.float32(0.82185066)), ('kunnen', np.float32(0.7908659)), ('zorgen', np.float32(0.78526646)), ('sterven', np.float32(0.7841965)), ('vertrekken', np.float32(0.7758652)), ('melden', np.float32(0.7685342)), ('slapen', np.float32(0.7669418)), ('kijken', np.float32(0.76553917)), ('ontsnappen', np.float32(0.76286817)), ('voorkomen', np.float32(0.7613393))]


In [69]:
# we test with a variant sentence
contextualised_distance = contextually_similar_verb(("spelen", "kind", "rol"))
verb_distance = top_similar_verbs("spelen", top_n=10)
print("Contextually similar verbs (core projection distance):", contextualised_distance[:10])
print("General similar verbs (factor cosine similarity):", verb_distance)

100%|██████████| 750/750 [00:02<00:00, 275.79it/s]


Contextually similar verbs (core projection distance): [('doen', np.float32(1.199327e-05)), ('horen', np.float32(3.3061701e-06)), ('maken', np.float32(3.2933697e-06)), ('weten', np.float32(2.4528126e-06)), ('lezen', np.float32(2.2725483e-06)), ('kennen', np.float32(2.1387773e-06)), ('vertellen', np.float32(1.9130694e-06)), ('houden', np.float32(1.8753162e-06)), ('zeggen', np.float32(1.8183682e-06)), ('schrijven', np.float32(1.6699565e-06))]
General similar verbs (factor cosine similarity): [('hoeven', np.float32(0.82185066)), ('kunnen', np.float32(0.7908659)), ('zorgen', np.float32(0.78526646)), ('sterven', np.float32(0.7841965)), ('vertrekken', np.float32(0.7758652)), ('melden', np.float32(0.7685342)), ('slapen', np.float32(0.7669418)), ('kijken', np.float32(0.76553917)), ('ontsnappen', np.float32(0.76286817)), ('voorkomen', np.float32(0.7613393))]


In [70]:
# we test with a variant sentence
contextualised_distance = contextually_similar_verb(("eten", "man", "ontbijt"))
verb_distance = top_similar_verbs("eten", top_n=10)
print("Contextually similar verbs (core projection distance):", contextualised_distance[:10])
print("General similar verbs (factor cosine similarity):", verb_distance)

100%|██████████| 750/750 [00:02<00:00, 263.24it/s]


Contextually similar verbs (core projection distance): [('nemen', np.float32(0.00070086285)), ('halen', np.float32(0.0006451825)), ('willen', np.float32(0.00048695714)), ('pak', np.float32(0.00043819708)), ('kopen', np.float32(0.0004152535)), ('laten', np.float32(0.00039753655)), ('brengen', np.float32(0.00038140648)), ('neem', np.float32(0.00033005758)), ('zetten', np.float32(0.00026319415)), ('zeggen', np.float32(0.00025996042))]
General similar verbs (factor cosine similarity): [('drinken', np.float32(0.83377236)), ('leren', np.float32(0.8111652)), ('regelen', np.float32(0.7217817)), ('stellen', np.float32(0.7048466)), ('praten', np.float32(0.69086295)), ('ontdekken', np.float32(0.65905976)), ('spelen', np.float32(0.6473193)), ('vangen', np.float32(0.64727527)), ('verkopen', np.float32(0.63754666)), ('betalen', np.float32(0.63650554))]


In [71]:
# we test with a variant sentence
contextualised_distance = contextually_similar_obj(("eten", "ik", "lunch"))
print("Contextually similar objects (core projection distance):", contextualised_distance[:10])
contextualised_distance = contextually_similar_subj(("eten", "ik", "lunch"))
print("Contextually similar subjects (core projection distance):", contextualised_distance[:10])

100%|██████████| 750/750 [00:02<00:00, 279.04it/s]


Contextually similar objects (core projection distance): [('spul', np.float32(2.1650438)), ('auto', np.float32(2.0178068)), ('hoed', np.float32(1.9304575)), ('koffer', np.float32(1.8133984)), ('paard', np.float32(1.8043268)), ('geweer', np.float32(1.7500315)), ('wapen', np.float32(1.7276564)), ('geld', np.float32(1.705741)), ('koffie', np.float32(1.6415782)), ('die', np.float32(1.5626419))]


100%|██████████| 750/750 [00:02<00:00, 279.16it/s]

Contextually similar subjects (core projection distance): [('~', np.float32(0.21070175)), ('je', np.float32(0.16869858)), ('we', np.float32(0.07082737)), ('u', np.float32(0.021903794)), ('hij', np.float32(0.004819145)), ('jij', np.float32(0.0004855504)), ('jezelf', np.float32(5.8081536e-05)), ('spoorweg', np.float32(3.2734613e-05)), ('uitweg', np.float32(1.8827799e-05)), ('iedereen', np.float32(4.512968e-07))]
